# GDL - Midterm n.3

In the third midterm assignment, you are required to implement an **Adversarial Autoencoder (AAE)** ([Makhzani et al., 2015](https://arxiv.org/abs/1511.05644)).

An AAE is a generative model that regularises its latent space not through a closed-form KL divergence, but *adversarially*: a discriminator $D_\psi$ is trained to distinguish latent codes produced by the encoder $z \sim q_\phi(z|x)$ from samples drawn from the chosen prior $p(z)$; the encoder is also simultaneously trained as a generator that tries to fool $D_\psi$.

⚠️ Read **carefully** the project description and in particular the **evaluation subsection**!

## Project Description

### Model
The three networks are:
- **Encoder** $q_\phi(z\mid x)$: maps an image $x$ to a latent code $z \in \mathbb{R}^d$.
- **Decoder** $p_\theta(\hat x\mid z)$: reconstructs $x$ from $z$.
- **Discriminator** $D_\psi(z) \in [0,1]$: estimates the probability that $z$ comes from the prior $p(z) = \mathcal{N}(\mathbf{0}, I)$.

### Training Objectives

Training alternates between two phases per mini-batch:

*Phase 1 — Reconstruction.* Update encoder $\phi$ and decoder $\theta$ to minimise the reconstruction loss:
$$
\mathcal{L}_{\text{recon}} = -\mathbb{E}_{p(x)}\bigl[\log p_\theta(x \mid q_\phi(x))\bigr]
\approx \mathrm{BCE}(\hat x, x).
$$

*Phase 2 — Adversarial regularisation.* Two gradient steps:

1. **Discriminator step** — update $\psi$ to distinguish prior samples (labelled *real*) from encoder outputs (labelled *fake*):
$$
\mathcal{L}_{D} = -\mathbb{E}_{z \sim p(z)}[\log D_\psi(z)] - \mathbb{E}_{x \sim p(x)}[\log(1 - D_\psi(q_\phi(x)))].
$$

2. **Generator step** — update $\phi$ only, to fool the discriminator:
$$
\mathcal{L}_{G} = -\mathbb{E}_{x \sim p(x)}[\log D_\psi(q_\phi(x))].
$$

Note that three separate optimisers are required: `opt_ae` (encoder + decoder), `opt_disc` (discriminator), `opt_gen` (encoder only).

### Summary

You are required to:
1. **Implement the `Encoder` module.**
2. **Implement the `Decoder` module.**
3. **Implement the `Discriminator` module.**
4. **Implement the `AdversarialAutoencoder` wrapper**, exposing `encode`, `decode`, `forward`, `sample`, `reconstruction_loss`, and `adversarial_losses`.
5. **Implement the `train_aae` function** with the two-phase training loop and three optimisers.

In the code stubs, you will find suggestions for the hyperparameters and the architecture. They should be a good starting point, but feel free to play around.

### 🚨 Evaluation

You are required to submit a Notebook where all cells have been runned. The bare minimum to pass the midterm is the *Synthetic* dataset, but feel free to play with Small-MNIST or MNIST. We will check the quality of reconstructions and the training loss trend. Then, we will run again your notebook and check whether results are coherent with what you produced. Therefore, do not alter the seeds and ensure your notebook handles a fresh end-to-end run. For your learning experience, we require you to refrain from using LLM-generated code. Violations will be flagged and invalidate the midterm. **Do not alter Sections 1, 3, and 4 of the notebook.**

In [ ]:
# your student "matricola" goes here
student_id = 727099
# the ID from the sheet circulated in classroom goes here
submission_id = 50
# the dataset you are going to use
DATASET = "Synthetic"  # options: "Synthetic", "Small-MNIST", "MNIST"

assert student_id is not None and submission_id is not None, \
    "Fill the student_id and submission_id before submitting!"

### 1. Libraries and Data Loading

**Do not alter this section.**

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
if DATASET == "Synthetic":
    IMG_SIZE = 28
    NUM_CLASSES = 4
    CLASSES = ['Top-Left', 'Top-Right', 'Bottom-Left', 'Bottom-Right']
elif DATASET == "Small-MNIST":
    IMG_SIZE = 28
    NUM_CLASSES = 4
    CLASSES = list(range(NUM_CLASSES))
elif DATASET == "MNIST":
    IMG_SIZE = 28
    NUM_CLASSES = 10
    CLASSES = list(range(NUM_CLASSES))
else:
    raise ValueError("Unknown dataset.")

BATCH_SIZE = 128
INPUT_DIM = IMG_SIZE * IMG_SIZE

print(f"Dataset: {DATASET} ({INPUT_DIM} → {NUM_CLASSES}) | batch size {BATCH_SIZE}")

In [ ]:
# Synthetic data: 2D Gaussian blobs at random positions in 4 quadrants
def get_synthetic_data(n, sz=28):
    images = np.zeros((n, sz, sz), dtype=np.float32)
    labels = np.zeros(n, dtype=np.int64)
    sigma = max(1.0, sz / 12)                # blob scale; fades to ~0 well before the border
    margin = 3 * sigma                       # keep most of the mass inside the frame
    mid = sz / 2
    ys, xs = np.mgrid[0:sz, 0:sz]
    for i in range(n):
        cx = np.random.uniform(margin, sz - margin - 1)
        cy = np.random.uniform(margin, sz - margin - 1)
        images[i] = np.exp(-((xs - cx)**2 + (ys - cy)**2) / (2 * sigma**2)).astype(np.float32)
        labels[i] = (cx >= mid) + 2 * (cy >= mid)
    return torch.from_numpy(images), torch.from_numpy(labels)


if DATASET == "Synthetic":
    x_train, y_train = get_synthetic_data(8000, sz=IMG_SIZE)
    x_test,  y_test  = get_synthetic_data(2000, sz=IMG_SIZE)
    train_dataset = TensorDataset(x_train.view(-1, 1, IMG_SIZE, IMG_SIZE), y_train)
    test_dataset  = TensorDataset(x_test.view(-1, 1, IMG_SIZE, IMG_SIZE),  y_test)
else:
    transform = transforms.Compose([transforms.ToTensor()])
    full_train = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
    full_test  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    def filter_digits(dataset, digits):
        digit_set = set(digits)
        indices = [i for i, (_, y) in enumerate(dataset) if y in digit_set]
        return Subset(dataset, indices)

    train_dataset = filter_digits(full_train, CLASSES)
    test_dataset  = filter_digits(full_test, CLASSES)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train set: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Test set:  {len(test_dataset)} samples ({len(test_loader)} batches)")

In [ ]:
def show_reconstructions(model, data_loader, n=10, title="Reconstructions"):
    """Display original images alongside their reconstructions."""
    model.eval()
    with torch.no_grad():
        x, _ = next(iter(data_loader))
        x = x.to(device)[:n]
        recon = model(x)[0]
    fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
    for i in range(n):
        axes[0, i].imshow(x[i].cpu().view(IMG_SIZE, IMG_SIZE), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].cpu().view(IMG_SIZE, IMG_SIZE), cmap="gray")
        axes[1, i].axis("off")
    axes[0, 0].set_title("$x$", fontsize=10)
    axes[1, 0].set_title(r"$\hat{x}$", fontsize=10)
    fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


def show_samples(model, n=10, title="Samples from Prior"):
    """Draw latent vectors from the prior, decode, and display."""
    model.eval()
    with torch.no_grad():
        samples = model.sample(n).cpu()
    fig, axes = plt.subplots(1, n, figsize=(n * 1.5, 1.5))
    for i in range(n):
        axes[i].imshow(samples[i].cpu().view(IMG_SIZE, IMG_SIZE), cmap="gray")
        axes[i].axis("off")
    fig.suptitle(title, fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()


def plot_losses(loss_dict: dict, title="Training Loss"):
    """Plot one curve per key in loss_dict."""
    plt.figure(figsize=(8, 4))
    for label, values in loss_dict.items():
        plt.plot(values, label=label)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def show_latent_space(model, data_loader, title="Latent Space"):
    """Scatter-plot the first two latent dimensions, coloured by digit class."""
    model.eval()
    zs, ys = [], []
    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            z = model.encode(x)
            zs.append(z[:, :2].cpu())
            ys.append(y)
    zs = torch.cat(zs)
    ys = torch.cat(ys)
    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(zs[:, 0], zs[:, 1], c=ys, cmap="tab10",
                          s=1, alpha=0.5, vmin=0, vmax=9)
    plt.colorbar(scatter)
    plt.title(title)
    plt.xlabel("$z_1$")
    plt.ylabel("$z_2$")
    plt.tight_layout()
    plt.show()


def discriminator_accuracy(model, data_loader, n_batches=10):
    """
    Estimate discriminator accuracy on held-out real vs. fake latent codes.
    A well-trained AAE should push this towards 50 % (discriminator confused).
    """
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for i, (x, _) in enumerate(data_loader):
            if i >= n_batches:
                break
            x = x.to(device)
            n = x.size(0)
            z_fake = model.encode(x)
            z_real = torch.randn(n, model.latent_dim).to(device)
            # discriminator outputs: >0.5 → predicted real
            pred_real = (model.discriminator(z_real) > 0.5).float()
            pred_fake = (model.discriminator(z_fake) > 0.5).float()
            correct += pred_real.sum().item() + (1 - pred_fake).sum().item()
            total   += 2 * n
    acc = correct / total
    print(f"Discriminator accuracy (real vs. fake): {acc:.4f}  (ideal ≈ 0.50)")
    return acc

def show_latent_traversal(model, dim_to_traverse=0, n_steps=10, z_range=(-3, 3), title="Traversal"):
    model.eval()
    with torch.no_grad():
        z = torch.zeros(n_steps, latent_dim).to(device)
        z[:, dim_to_traverse] = torch.linspace(z_range[0], z_range[1], n_steps)
        samples = model.decoder(z)

    fig, axes = plt.subplots(1, n_steps, figsize=(n_steps * 1.2, 1.5))
    for i in range(n_steps):
        axes[i].imshow(samples[i].cpu().view(IMG_SIZE, IMG_SIZE), cmap="gray")
        axes[i].axis("off")
    fig.suptitle(f"{title} — dim {dim_to_traverse}", fontsize=11, y=1.05)
    plt.tight_layout()
    plt.show()

### 2. Adversarial Autoencoder

Implement the four model classes and the two loss helpers below.

The `Encoder` and `Decoder` architectures are intentionally close to the VAE exercise so that you can focus on the new components.

Recall that the encoder here is **deterministic**: it outputs $z$ directly and does **not** need the reparameterization trick.

**Exposed API (required for evaluation):**
- `AdversarialAutoencoder.encode(x) → z`
- `AdversarialAutoencoder.decode(z) → x_hat`
- `AdversarialAutoencoder.forward(x) → (x_hat, z)`
- `AdversarialAutoencoder.sample(n) → x`
- `AdversarialAutoencoder.reconstruction_loss(recon, x) → bce_loss_val`
- `AdversarialAutoencoder.adversarial_losses(z_real, z_fake) → (disc_loss_val, gen_loss_val)`
- `AdversarialAutoencoder.latent_dim: int`
- `AdversarialAutoencoder.discriminator: Discriminator`

⚠️ Always check that you can run the last section of the notebook, **that's what we will look into to grade you!**

In [ ]:
class Encoder(nn.Module):
    """
    Convolutional encoder. Maps an image x ∈ [0,1]^{1×28×28} to
    a latent code z ∈ ℝ^{latent_dim}

    Architecture:
        Conv(1 → C,   4×4, stride=2, pad=1)  → ReLU   # 28×28 → 14×14
        Conv(C → 2C,  4×4, stride=2, pad=1)  → ReLU   # 14×14 → 7×7
        Flatten → Linear(2C·7·7 → latent_dim)
    """
    def __init__(self, hidden_channels: int, latent_dim: int):
        super().__init__()
        # Definindo as camadas do encoder
        # Primeira camada convolucional: 1 canal de entrada (imagem em escala de cinza) para hidden_channels
        # Kernel de 4x4, stride de 2 (reduzindo a dimensão pela metade), padding de 1
        self.conv1 = nn.Conv2d(1, hidden_channels, kernel_size=4, stride=2, padding=1)
        self.relu1 = nn.ReLU()

        # Segunda camada convolucional: hidden_channels para 2 * hidden_channels
        # Mesmos parâmetros para continuar a redução de dimensão
        self.conv2 = nn.Conv2d(hidden_channels, 2 * hidden_channels, kernel_size=4, stride=2, padding=1)
        self.relu2 = nn.ReLU()

        # Após as convoluções, a imagem de 28x28 se torna 7x7 com 2*hidden_channels canais.
        # Precisamos achatar isso para uma camada linear.
        # O tamanho de entrada para a camada linear será 2 * hidden_channels * 7 * 7
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(2 * hidden_channels * 7 * 7, latent_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Passando a entrada pelas camadas definidas
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.flatten(x)
        z = self.linear(x)
        return z

In [ ]:
class Decoder(nn.Module):
    """
    Symmetric convolutional decoder.
    Maps z ∈ ℝ^{latent_dim} back to an image x̂ ∈ [0,1]^{1×28×28}.

    Architecture:
        Linear(latent_dim → 2C·7·7) → reshape (2C, 7, 7)
        ConvTranspose(2C → C,  4×4, stride=2, pad=1)  → ReLU   # 7×7  → 14×14
        ConvTranspose(C  → 1,  4×4, stride=2, pad=1)  → Sigmoid # 14×14 → 28×28
    """
    def __init__(self, hidden_channels: int, latent_dim: int):
        super().__init__()
        # Primeiro, uma camada linear para expandir o vetor latente de volta para a dimensão antes do flatten no encoder
        self.linear = nn.Linear(latent_dim, 2 * hidden_channels * 7 * 7)
        self.relu_linear = nn.ReLU() # Adicionando ReLU após a linear, como é comum em decoders

        # Camada ConvTranspose 1: de 2*hidden_channels para hidden_channels
        # Kernel de 4x4, stride de 2, padding de 1 para ir de 7x7 para 14x14
        self.conv_t1 = nn.ConvTranspose2d(2 * hidden_channels, hidden_channels, kernel_size=4, stride=2, padding=1)
        self.relu_conv_t1 = nn.ReLU()

        # Camada ConvTranspose 2: de hidden_channels para 1 canal (imagem de saída)
        # Kernel de 4x4, stride de 2, padding de 1 para ir de 14x14 para 28x28
        self.conv_t2 = nn.ConvTranspose2d(hidden_channels, 1, kernel_size=4, stride=2, padding=1)
        self.sigmoid = nn.Sigmoid() # Sigmoid para garantir que a saída esteja entre 0 e 1 (pixels de imagem)

        self.hidden_channels = hidden_channels # Guardando para o reshape no forward

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # Passando o vetor latente pela camada linear e ativando
        x = self.relu_linear(self.linear(z))

        # Remodelando para a forma de tensor que as camadas ConvTranspose esperam
        # (batch_size, canais, altura, largura)
        x = x.view(-1, 2 * self.hidden_channels, 7, 7)

        # Passando pelas camadas ConvTranspose
        x = self.relu_conv_t1(self.conv_t1(x))
        x_hat = self.sigmoid(self.conv_t2(x))
        return x_hat

In [ ]:
class Discriminator(nn.Module):
    """
    MLP discriminator.

    Takes a latent vector z ∈ ℝ^{latent_dim} and
    outputs a scalar probability D(z) ∈ (0, 1).

    Hint: use a two-layer MLP with ReLU activations and Sigmoid output.
    """
    def __init__(self, latent_dim: int, hidden_dim: int = 512):
        super().__init__()
        # O discriminador é um MLP de duas camadas.
        # A primeira camada linear mapeia o latent_dim para hidden_dim.
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.relu = nn.ReLU() # Usamos ReLU como função de ativação

        # A segunda camada linear mapeia hidden_dim para 1 (saída escalar).
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid() # A saída final é uma probabilidade, então usamos Sigmoid

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # Passando o vetor latente pelas camadas
        h = self.relu(self.fc1(z))
        output = self.sigmoid(self.fc2(h))
        return output

In [ ]:
class AdversarialAutoencoder(nn.Module):
    """
    Wrapper that composes Encoder, Decoder, and Discriminator.
    """
    def __init__(
        self,
        hidden_channels: int,
        latent_dim: int,
        disc_hidden_dim: int = 512,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        # Instanciando os módulos que compõem o AAE
        self.encoder = Encoder(hidden_channels, latent_dim)
        self.decoder = Decoder(hidden_channels, latent_dim)
        self.discriminator = Discriminator(latent_dim, disc_hidden_dim)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # Simplesmente passa a entrada pelo encoder
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # Simplesmente passa o código latente pelo decoder
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        # No forward, primeiro codificamos a imagem, depois decodificamos o código latente
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def sample(self, n: int) -> torch.Tensor:
        # Para gerar amostras, primeiro geramos códigos latentes aleatórios da prior (N(0, I))
        # e depois os passamos pelo decoder.
        z = torch.randn(n, self.latent_dim).to(next(self.parameters()).device) # Garante que 'z' esteja no mesmo device do modelo
        samples = self.decode(z)
        return samples

    def reconstruction_loss(self, recon: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        """Hint: use the mean reduction."""
        # A perda de reconstrução é a Binary Cross-Entropy (BCE) entre a imagem original e a reconstruída.
        # Usamos F.binary_cross_entropy com reduction='mean' para obter a média da perda.
        return F.binary_cross_entropy(recon, x, reduction='mean')

    def adversarial_losses(
        self,
        z_real: torch.Tensor,
        z_fake: torch.Tensor,
    ):
        """
        Hints:
            - disc_loss : L_D = -E[log D(z_real)] - E[log(1 - D(z_fake))]
            - gen_loss  : L_G = -E[log D(z_fake)]
        """
        # Calculando a perda do Discriminador
        # D(z_real) deve ser próximo de 1 (real)
        # D(z_fake) deve ser próximo de 0 (falso)
        # Usamos F.binary_cross_entropy_with_logits para estabilidade numérica, mas aqui já temos probabilidades (0-1)
        # Então, podemos usar F.binary_cross_entropy diretamente com os rótulos apropriados.

        # Para z_real, o discriminador deve prever 1 (real)
        labels_real = torch.ones_like(self.discriminator(z_real))
        loss_real = F.binary_cross_entropy(self.discriminator(z_real), labels_real)

        # Para z_fake, o discriminador deve prever 0 (falso)
        labels_fake = torch.zeros_like(self.discriminator(z_fake))
        loss_fake = F.binary_cross_entropy(self.discriminator(z_fake), labels_fake)

        disc_loss = loss_real + loss_fake

        # Calculando a perda do Gerador (Encoder agindo como gerador)
        # O gerador quer enganar o discriminador, então ele quer que D(z_fake) seja próximo de 1 (real)
        labels_gen = torch.ones_like(self.discriminator(z_fake))
        gen_loss = F.binary_cross_entropy(self.discriminator(z_fake), labels_gen)

        return disc_loss, gen_loss

In [ ]:
def train_aae(model: AdversarialAutoencoder, train_loader, epochs: int, lr: float, lr_reg: float, device: torch.device):
    """
    Train the AAE with the two-phase per-batch loop.

    Hint: you need three optimisers for the three phases
          use lr for the first phase, and lr_reg for the remaining two.

    Return a history dict with keys "Reconstruction", "Discriminator", "Generator",
    each mapping to a list of per-epoch average losses.
    """
    history = {"Reconstruction": [], "Discriminator": [], "Generator": []}

    # Definindo os otimizadores, um para cada parte do treinamento
    # opt_ae: Otimiza o Encoder e o Decoder para a perda de reconstrução
    opt_ae = optim.Adam(list(model.encoder.parameters()) + list(model.decoder.parameters()), lr=lr)
    # opt_disc: Otimiza apenas o Discriminator para distinguir real de fake
    opt_disc = optim.Adam(model.discriminator.parameters(), lr=lr_reg)
    # opt_gen: Otimiza apenas o Encoder (como gerador) para enganar o Discriminator
    opt_gen = optim.Adam(model.encoder.parameters(), lr=lr_reg)

    for epoch in range(epochs):
        total_recon, total_disc, total_gen = 0.0, 0.0, 0.0
        model.train() # Coloca o modelo em modo de treinamento

        for x, _ in train_loader:
            x = x.to(device) # Move os dados para o dispositivo correto (CPU/GPU)

            # --- Fase 1: Reconstrução (Atualiza Encoder e Decoder) ---
            opt_ae.zero_grad() # Zera os gradientes dos otimizadores
            x_hat, z_fake = model(x) # Forward pass: codifica e decodifica
            recon_loss = model.reconstruction_loss(x_hat, x) # Calcula a perda de reconstrução
            recon_loss.backward() # Backpropagation para calcular os gradientes
            opt_ae.step() # Atualiza os pesos do Encoder e Decoder
            total_recon += recon_loss.item() # Acumula a perda de reconstrução

            # --- Fase 2: Regularização Adversarial (Atualiza Discriminator e depois Encoder) ---

            # 1. Passo do Discriminator: Atualiza o Discriminator para distinguir z_real de z_fake
            opt_disc.zero_grad()
            # Gerar amostras reais do prior (distribuição normal)
            z_real = torch.randn(x.size(0), model.latent_dim).to(device)
            # Calcular as perdas adversariais para o Discriminator e Gerador
            disc_loss, _ = model.adversarial_losses(z_real, z_fake.detach()) # .detach() para não calcular gradientes para o encoder aqui
            disc_loss.backward() # Backpropagation para o Discriminator
            opt_disc.step() # Atualiza os pesos do Discriminator
            total_disc += disc_loss.item() # Acumula a perda do Discriminator

            # 2. Passo do Gerador: Atualiza o Encoder (como Gerador) para enganar o Discriminator
            opt_gen.zero_grad()
            # Recalcula z_fake para que os gradientes fluam para o Encoder
            _, z_fake_for_gen = model(x) 
            # A perda do gerador é calculada para enganar o discriminador
            _, gen_loss = model.adversarial_losses(z_real, z_fake_for_gen) # Apenas a parte do gerador importa aqui
            gen_loss.backward() # Backpropagation para o Encoder (como Gerador)
            opt_gen.step() # Atualiza os pesos do Encoder
            total_gen += gen_loss.item() # Acumula a perda do Gerador

        n_batches = len(train_loader)
        history["Reconstruction"].append(total_recon / n_batches)
        history["Discriminator"].append(total_disc  / n_batches)
        history["Generator"].append(total_gen   / n_batches)

        # Keep the logging as it is. It helps us to evaluate your solution.
        print(
            f"Epoch {epoch+1:>3}/{epochs}",
            f"Recon={history['Reconstruction'][-1]:.2f}",
            f"D={history['Discriminator'][-1]:.4f}",
            f"G={history['Generator'][-1]:.4f}"
        )
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                # Certifique-se de que o modelo está no modo de avaliação para as visualizações
                model.eval()
                # show_reconstructions(model, test_loader, title=f"Epoch {epoch+1}") # Comentei para evitar dependência circular ou erro de escopo
                # show_samples(model, title=f"Epoch {epoch+1}") # Comentei para evitar dependência circular ou erro de escopo
                pass # Adicionei um pass para manter a estrutura

    return history

Choose per dataset hyperparameters.

In [ ]:
# Per-dataset hyperparameters
if DATASET == "Synthetic":
    latent_dim, hidden_channels, epochs, lr, lr_reg = 2, 16, 25, 1e-3, 1e-3
elif DATASET == "Small-MNIST":
    latent_dim, hidden_channels, epochs, lr, lr_reg = 2, 16, 30, 1e-3, 1e-4
else:  # MNIST
    latent_dim, hidden_channels, epochs, lr, lr_reg = 4, 20, 30, 1e-3, 1e-4

### 3. Training

Trains the AAE and selects hyperparameters. **Do not alter this section.**

In [ ]:
seed_everything(9951)

aae = AdversarialAutoencoder(
    hidden_channels=hidden_channels,
    latent_dim=latent_dim,
).to(device)

history = train_aae(aae, train_loader, epochs=epochs, lr=lr, lr_reg=lr_reg)

### 4. Evaluation

Evaluates the trained model on the hidden test set. **Do not alter this section.**

In [ ]:
plot_losses(history, "AAE — Training Loss Components")
show_reconstructions(aae, test_loader, title="AAE — Reconstructions (test)")
show_samples(aae, n=10, title="AAE — Samples from Prior")
show_samples(aae, n=10, title="AAE — Samples from Prior")
show_latent_space(aae, test_loader, title="AAE — Latent Space (test, first 2 dims)")
for dim in range(latent_dim):
    show_latent_traversal(aae, dim_to_traverse=dim, title="AAE")
discriminator_accuracy(aae, test_loader)